# RPT API — Interview Playground

## Setup

Before you start:
1. Run the **Setup** and **Load Data** cells
2. Each task is independent — you can work on them in any order

> **Note:** The API has a rate limit of 10 requests per hour. Plan your calls before running them.

In [15]:
import requests
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

with open('config.json') as f:
    _cfg = json.load(f)

API_TOKEN = _cfg['RPT_TOKEN']
API_URL   = _cfg['RPT_API_URL']
HEADERS   = {'Authorization': f'Bearer {API_TOKEN}', 'Content-Type': 'application/json'}

if API_TOKEN == 'YOUR_TOKEN_HERE':
    print('Token not set — open config.json and replace YOUR_TOKEN_HERE')
else:
    print('Token loaded ✓')
    print('API URL:', API_URL)

Token loaded ✓
API URL: https://rpt.cloud.sap/api/predict


---

## Dataset: Montgomery County Employee Salaries

Salary records for public employees of Montgomery County, MD (USA), 2016.

| Column | Type | Description |
|---|---|---|
| `gender` | categorical | `M` / `F` |
| `department` | categorical | Short department code (e.g. `POL`, `FRS`) |
| `department_name` | categorical | Full department name |
| `division` | categorical | Sub-unit within the department |
| `assignment_category` | categorical | Employment type — `Fulltime-Regular`, `Parttime-Regular`, etc. |
| `employee_position_title` | categorical | Job title as recorded in HR systems |
| `date_first_hired` | string | Date the employee first joined (MM/DD/YYYY) |
| `year_first_hired` | numeric | Year extracted from `date_first_hired` |
| `current_annual_salary` | numeric | Gross annual salary in USD |

In [16]:
df = pd.read_csv('employee_salaries.csv')
print(df.shape)
df.head()

(9228, 10)


,gender,department,department_name,division,assignment_category,employee_position_title,date_first_hired,year_first_hired,current_annual_salary,is_eval
0,M,DLC,Department of Liquor Control,Clarksburg Village,Parttime-Regular,Liquor Store Clerk I,09/03/1991,1991,54022.26,0
1,F,POL,Department of Police,FSB Traffic Division Automated Traffic Enforce...,Fulltime-Regular,Police Aide,02/05/2007,2007,46626.06,0
2,F,COR,Correction and Rehabilitation,DS MCCF Case Managers Unit 1,Fulltime-Regular,Correctional Specialist IV,01/06/1997,1997,97153.42,0
3,F,PRO,Office of Procurement,Procurement Director,Fulltime-Regular,Director Office of Procurement,01/28/2014,2014,190550.00,0
4,M,HHS,Department of Health and Human Services,Income Supports,Fulltime-Regular,Income Assistance Program Specialist II,02/01/1999,1999,73819.32,0


---

## Background

You receive this message from a senior HR business partner:

> *"Hey, so leadership has been pushing us on pay equity for a while. We've had complaints from employees who feel their salary doesn't reflect their role or experience. I heard your team has access to some new AI prediction tool. Could you use it on our employee data and come back with something? Ideally we'd want to understand if the model can tell us what someone 'should' be earning. Oh and Finance is also asking whether part-time staff are being treated consistently. No pressure, but leadership wants a slide by Friday."*

The dataset above is what you have available. The tasks below guide your analysis.

---

## API Reference

The helper functions below (`get_payload`, `get_response`, `get_predictions`) already implement the API calls. You can use them directly without writing request logic yourself.

For the full API documentation see **[api_reference.ipynb](api_reference.ipynb)**.

In [ ]:
def get_payload(context_df, eval_df_batch, target_col='current_annual_salary'):
    context_rows = json.loads(context_df.to_json(orient='records'))
    eval_copy = eval_df_batch.copy()
    eval_copy[target_col] = '[PREDICT]'
    eval_rows = json.loads(eval_copy.to_json(orient='records'))
    return {'rows': context_rows + eval_rows}

def get_response(payload):
    response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)
    if not response.ok:
        print(f'Error {response.status_code}: {response.text}')
    response.raise_for_status()
    return response.json()

def get_predictions(result, target_col='current_annual_salary'):
    return pd.json_normalize(
        result['prediction']['predictions'],
        record_path=target_col
    )['prediction'].astype(float).values

def plot_distribution(series, title='Salary distribution'):
    """Plot a histogram with median line."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(series.dropna(), bins=60, edgecolor='none')
    ax.axvline(series.median(), color='red', linestyle='--', label=f'Median ${series.median():,.0f}')
    ax.set_title(title)
    ax.set_xlabel('Annual salary (USD)')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_predicted_vs_actual(y_true, y_pred, title='Predicted vs. actual salary'):
    """Scatter plot of predictions against ground truth with a perfect-fit line."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_true, y_pred, alpha=0.7)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', label='perfect fit')
    ax.set_xlabel('Actual salary (USD)')
    ax.set_ylabel('Predicted salary (USD)')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

---

## Task 1 — Data Audit

Before building anything, explore the dataset.

- What does the salary distribution look like?
- Are there columns you would **not** use for prediction, and why?
- Are there any columns that could cause unexpected problems when working with the API?

Summarize your findings in a markdown cell.

In [ ]:
df = pd.read_csv('employee_salaries.csv')  # fresh load — do not depend on earlier cells

*Your findings here.*

---

## Task 2 — Feature Engineering

Engineer at least one new feature that you believe adds predictive value.

- Explain why you created it
- Remove any columns you consider redundant or harmful
- Store the result in `df_clean`

In [ ]:
df = pd.read_csv('employee_salaries.csv')  # fresh load
df_clean = df.copy()

---

## Task 3 — First Prediction

Use the RPT API to predict the salary of **one employee** from the eval set.

- How many context rows do you use, and why?
- Print the predicted value and the actual salary
- Is the prediction reasonable? Comment briefly.

In [ ]:
df = pd.read_csv('employee_salaries.csv')

context_df = df[df['is_eval'] == 0].drop(columns='is_eval')
eval_df    = df[df['is_eval'] == 1].drop(columns='is_eval')

---

## Task 4 — Model Evaluation

Evaluate the model on all employees in the eval set.

- Write your own evaluation logic
- Compute at least two metrics
- Interpret the results in business terms: is this model useful for the stakeholder's goal?
- Plot predicted vs. actual salaries

In [ ]:
df = pd.read_csv('employee_salaries.csv')

context_df = df[df['is_eval'] == 0].drop(columns='is_eval')
eval_df    = df[df['is_eval'] == 1].drop(columns='is_eval')

*Your interpretation here.*

---

## Task 5 — Salary Patterns & Pay Equity

The stakeholder asked whether salaries are "fair and consistent", and Finance specifically wants to know whether part-time staff are being treated consistently.

- Use the data to find at least two interesting salary patterns worth flagging
- The dataset doesn't have everything you'd need to fully test part-time consistency. What data would you need, and how would the analysis look?

In [ ]:
df = pd.read_csv('employee_salaries.csv')  # fresh load